# 02 - Qwen2.5-0.5B QLoRA fine-tuning (Google Colab)

**Runtime -> Change runtime type -> T4 GPU** (free tier is enough), then run top to bottom.

Trains the sentiment classifier end-to-end with the Hugging Face stack and saves the
adapter so the benchmark/serving can pick it up.

## 0. Environment check

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Get the code + data

In [ ]:
# Option A: clone your fork (recommended once the repo is on GitHub)
# !git clone https://github.com/<your-user>/hotel-review-nlp.git
# %cd hotel-review-nlp

# Option B: upload a zip of the repo through the Colab sidebar
# !unzip -q hotel-review-nlp.zip -d . && %cd hotel-review-nlp

In [ ]:
!pip install -q -e '.[dev]' && pip install -q -e '.[llm]'

In [ ]:
# Upload data/processed/*.parquet from your machine (produced by `make data`),
# or rebuild them here if data/raw/booking_reviews_515k.csv is uploaded too.
import os
assert os.path.exists("data/processed/train.parquet"), "upload data/processed first"
assert os.path.exists("data/processed/dev.parquet"), "upload data/processed first" 

## 2. Train (QLoRA: 4-bit NF4 base + LoRA adapters)

In [ ]:
# Mirrors configs/qlora_qwen.yaml; ~25-40 min on a free T4 for 20k examples / 1 epoch
!python -m reviewnlp.llm.train_qlora --config configs/qlora_qwen.yaml

## 3. Sanity-check the adapter

In [ ]:
from reviewnlp.llm.predict import predict_qwen_qlora

preds = predict_qwen_qlora("runs/qwen_qlora/adapter", [
    "The staff was wonderful and the breakfast excellent.",
    "Filthy room, broken AC, rude reception. Never again.",
])
print(list(preds))  # expect ['positive', 'negative']

## 4. Save the adapter (optional: Google Drive)

In [ ]:
# from google.colab import drive  # uncomment on Colab
# drive.mount("/content/drive")
# !cp -r runs/qwen_qlora/adapter /content/drive/MyDrive/qwen_qlora_hotel_adapter

## 5. Next steps
- Run the unified benchmark: `python -m reviewnlp.evaluation.benchmark --config configs/baselines.yaml`
- Serve it: `MODEL_TYPE=qwen_qlora MODEL_PATH=runs/qwen_qlora/adapter uvicorn reviewnlp.serving.app:app`